In [ ]:

# Figure 1 B,D

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime
import math
from matplotlib.pyplot import cm
import statsmodels.api as sm
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportions_ztest
from paths import DATA_DIR, fig_dir

In [ ]:
# Custom schema
from alison_rlmodel import BehaviorModelResults # Changes wd for jl code


In [ ]:
from plot_content import *
from find_my_data import *
from plot_rlmodel import *
from plot_figs_summary_metrics import get_all_rat_stable_nwb_file_names

### Figure formatting

In [ ]:
from fig_helpers import *

set_figure_defaults()

fig_path = fig_dir('figs26')

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

In [ ]:

save_fig = False

### Load data

In [ ]:
# Parameters
behavior_model_params_name = 'default_hmm_0623' #Any model ok for behavior, be specific if plot dvs

position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

In [ ]:
# Load data
out_path = f'{DATA_DIR}/big_df_pkls/'
today_now = '20230616' #Match loaded model params
subject_ids = ['j16', 'chimi','senor','wilbur', 'peanut']
is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
big_dfs = {}
for subject_id in subject_ids:
    try:
        if subject_id == 'senor':
            senor_big_df = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        if subject_id == 'chimi':
            chimi_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        if subject_id == 'wilbur':
            wilbur_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        if subject_id == 'peanut':
            peanut_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        if subject_id == 'j16':
            j16_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_uncertainty_'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

In [ ]:
all_nwb_file_names_dict, stable_nwb_file_names_dict = get_all_rat_stable_nwb_file_names(big_dfs,)

stable_nwb_file_names_dict['peanut'] = stable_nwb_file_names_dict['peanut'][0:-1]

all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    try:
        df = big_dfs[subject_id]
    except:
        df = big_dfs[str.capitalize(subject_id)]
    stable_nwbs = stable_nwb_file_names_dict[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_nwbs)]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable

for subject_id in subject_ids:
    df = all_rat_big_dfs_stable[subject_id]
    p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    all_rat_big_dfs_stable[subject_id] = df[~df[p_rew_cols].eq(df['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:
# Get behavior model only info
behavior_model_params_name = 'default_hmm_0623'
subject_ids = ['j16', 'chimi','senor','wilbur', 'peanut']


hmm_results = {}
for subject_id in subject_ids:
    hmm_result = (BehaviorModelResults() & {'behavior_model_params_name': behavior_model_params_name, 'subject_id': subject_id}).fetch1_dataframe()
    hmm_results[subject_id] = hmm_result

hmm_results_stable = {}
for subject_id in subject_ids:
    df = hmm_results[subject_id]
    stable_nwbs = stable_nwb_file_names_dict[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_nwbs)]
    hmm_results_stable[subject_id] = df_stable

### Inspect

In [ ]:
hmm_results_stable['j16']

In [ ]:
n_total=0
for subject_id in subject_ids:
    print(f'{subject_id}, n= {len(hmm_results_stable[subject_id])} trials')
    n_total += len(hmm_results_stable[subject_id])
print(n_total)

In [ ]:
# Load behavior model info without the 100all 50all session

behavior_model_params_name = 'default_hmm_0623'
subject_ids = ['j16', 'chimi','senor','wilbur', 'peanut']


hmm_results = {}
for subject_id in subject_ids:
    hmm_result = (BehaviorModelResults() & {'behavior_model_params_name': behavior_model_params_name, 'subject_id': subject_id}).fetch1_dataframe()
    hmm_results[subject_id] = hmm_result

hmm_results_stable = {}
for subject_id in subject_ids:
    df = hmm_results[subject_id]
    stable_nwbs = stable_nwb_file_names_dict[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_nwbs)]
    hmm_results_stable[subject_id] = df_stable

for subject_id in subject_ids:
    df = hmm_results_stable[subject_id]
    #p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    hmm_results_stable[subject_id] = df[np.logical_and(df['contingency']!=100100100100100100, df['contingency']!=505050505050)]

In [ ]:
n_total=0
for subject_id in subject_ids:
    print(f'{subject_id}, n= {len(hmm_results_stable[subject_id])} trials')
    n_total += len(hmm_results_stable[subject_id])
print(n_total)

### Functions

In [ ]:
def plot_grid_dots_new2(dataframe, figsize=(20,8), show_switches=True, show_switch_arrows_top=False, show_switch_arrows_bottom=False, plot_date=None, epochs=None, max_n_trials=None, min_n_trials=0,
                        save_fig=False, fig_path='', aspect_ratio_sq=True, dot_size=70,dot_line_thickness=2.5):
    # Define a color map for stems
    color_map = {'A': 'green', 'B': 'darkblue', 'C': 'mediumorchid'}
    
    # Filter by date if provided (taking into account the integer format yyyymmdd)
    if plot_date:
        dataframe = dataframe[dataframe['date'] == plot_date]
    
    # Filter by epochs if provided
    if epochs:
        dataframe = dataframe[dataframe['epoch'].isin(epochs)]
    
    if dataframe.empty:
        print("No data matches the provided criteria.")
        return
    
    # Filter by num trials
    if max_n_trials:
        dataframe = dataframe.iloc[min_n_trials:max_n_trials]

    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)
    if aspect_ratio_sq:
        ax.set_aspect('equal')  # Ensure equal aspect ratio

    # Draw the light grey grid lines
    for x in np.arange(0, len(dataframe)+1):
        ax.axvline(x, ymin=0, ymax=.85, color='lightgrey', linewidth=0.5, zorder=0)  # vertical lines with adjusted ymax ymax 5.5
#         plt.vline(x, ymin=0, ymax=6, color='lightgrey', linewidth=0.5, zorder=0)  
    for y in np.arange(0, 7):
        ax.axhline(y, color='lightgrey', linewidth=0.5, zorder=0)  # horizontal lines

    # Detecting epoch, contingency changes, and stem switches
    for i in range(1, len(dataframe)):
        if dataframe['epoch'].iloc[i] != dataframe['epoch'].iloc[i-1]:
            ax.axvline(i, color='black', linewidth=2)
        if dataframe['contingency'].iloc[i] != dataframe['contingency'].iloc[i-1]:
            ax.axvline(i, color='black', linewidth=1, linestyle='-')
        if show_switches and dataframe['stem'].iloc[i] != dataframe['stem'].iloc[i-1]:
            ax.axvline(i, color='red', linewidth=.7)
        if show_switch_arrows_top and dataframe['stem'].iloc[i] != dataframe['stem'].iloc[i-1]:
            ax.plot(i, 6.5, 'v', color='red', markersize=2) #6.5
        if show_switch_arrows_bottom and dataframe['stem'].iloc[i] != dataframe['stem'].iloc[i-1]:
            ax.plot(i, -.5, '^', color='red', markersize=3) #-.5

    # Plot each dot
    for i, (index, row) in enumerate(dataframe.iterrows()):
        edgecolor = color_map[row['stem']]
        facecolor = edgecolor if row['reward'] == 1 else 'white'  # Filled if rewarded
        ax.scatter(i + 0.5, row['leaf'] - 0.5, color=facecolor, edgecolor=edgecolor, s=dot_size, linewidth=dot_line_thickness)

    #ax.tick_params(left=False, bottom=False, labelleft=True, labelbottom=False)

    # Adjust labels and styles
    ax.set_yticks(np.arange(0.5,6.5))
    ax.set_yticklabels(np.arange(1, 7), fontsize=7) #fontsize=15)
    ax.set_ylabel('Port', ) #fontsize=20)
    ax.set_xlabel('Trial', )#fontsize=20)
    ax.set_xticks(range(0,len(dataframe),20), range(0,len(dataframe),20)) #, fontsize=15)
    ax.set_title(f'{plot_date} {epochs}', fontsize=4)
    
    ax.set_xlim(0, len(dataframe))
    ax.set_ylim(0, 7) #6 # Adjusting the ylim to give space for the triangles at the top
    plt.tight_layout()

    # Adjust plot outline color
    for spine in ax.spines.values():
        spine.set_edgecolor('lightgrey')
        spine.set_linewidth(0.5)
    ax.spines['top'].set_visible(False)

    
    if save_fig:
        fig_name = f'tall_fmt_{subject_id}_{plot_date}_{epochs[0]}_{min_n_trials}_{max_n_trials}_{show_switches}_{show_switch_arrows_top}_{show_switch_arrows_bottom}_{figsize[0]}_{figsize[1]}_{aspect_ratio_sq}_{dot_size}_{dot_line_thickness}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)       
        
    plt.show()

In [ ]:
def load_trialData(data_dict, plot_rat):
    trialData = data_dict[plot_rat]
    trialData['QA'] = trialData[['Q1','Q2']].mean(axis=1) #np.mean([trialData['Q1'],trialData['Q2']])
    trialData['QB'] = trialData[['Q3','Q4']].mean(axis=1)
    trialData['QC'] = trialData[['Q5','Q6']].mean(axis=1)
    trialData = trialData.reset_index()
    return trialData

def compute_stay_switch_values(row):
    leaves = range(1, 7)
    if row['SstemOption'] == 'A':
        Sleaves = ['1','2']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        GstemOption = ['B','C']
        SstemOptionBiased = '1'
        GstemOptionBiased = ['2','3']
    elif row['SstemOption'] == 'B':
        GstemOption = ['A','C']
        Sleaves = ['3','4']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        SstemOptionBiased = '2'
        GstemOptionBiased = ['1','3']
    elif row['SstemOption'] == 'C':
        GstemOption = ['A','B']
        Sleaves = ['5','6']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        SstemOptionBiased = '3'
        GstemOptionBiased = ['1','2']
    else:
        # Default values if 'SstemOption' is not A, B, or C
        Sleaves, Gleaves, GstemOption, SstemOptionBiased, GstemOptionBiased = np.nan, np.nan, np.nan, np.nan, np.nan
    return Sleaves, Gleaves, GstemOption, SstemOptionBiased, GstemOptionBiased

# Function to handle NaNs in SstemOption and perform mapping
def map_sstemoption(row, option_column, map_dict):
    option = row[option_column]
    if pd.isna(option) or option not in map_dict:
        return np.nan
    else:
        return row[map_dict[option]]

def get_log_reg_data_stay_switch(data_dict, subject_id):
    trialData = load_trialData(data_dict, subject_id)
    trialData['SstemOption'] = trialData.groupby(by = ['nwb_file_name', 'epoch',])['stem'].shift(1)
    new_cols = trialData.apply(compute_stay_switch_values, axis=1)
    trialData[['Sleaves', 'Gleaves', 'GstemOption', 'SstemOptionBiased', 'GstemOptionBiased']] = pd.DataFrame(new_cols.tolist(), index=trialData.index)

    options_map = {'A': 'QA', 'B': 'QB', 'C': 'QC'}
    options_map_biased = {'1': 'Qstem1', '2': 'Qstem2', '3': 'Qstem3'}
    trialData['qSmean'] = trialData.apply(map_sstemoption, axis=1, args=('SstemOption', options_map))
    trialData['qSmean_biased'] = trialData.apply(map_sstemoption, axis=1, args=('SstemOptionBiased', options_map_biased))

    # For qGmean and qGmean_biased
    trialData['qGmean'] = trialData.apply(
        lambda row: np.mean([row['Q' + val] for val in row['GstemOption']]) if isinstance(row['GstemOption'], list) else np.nan,
        axis=1)
    trialData['qGmean_biased'] = trialData.apply(
        lambda row: np.mean([row['Qstem' + val] for val in row['GstemOptionBiased']]) if isinstance(row['GstemOptionBiased'], list) else np.nan,
        axis=1)
    # calculate values for log reg
    trialData['stem_switch'] = (trialData['SstemOption']!=trialData['stem'])
    trialData['SstemOption'][trialData['SstemOption'].isna()] = np.nan
    trialData['qSminusG'] = trialData['qSmean']-trialData['qGmean']
    trialData['qSminusG_biased'] = trialData['qSmean_biased']-trialData['qGmean_biased']
    trialData['not_switch'] = ~trialData['stem_switch']
    # switch-stay instead of stay-switch
    trialData['qGminusS'] = trialData['qGmean']-trialData['qSmean']
    trialData['qGminusS_biased'] = trialData['qGmean_biased']-trialData['qSmean_biased']
    
    trialData['contingency_shifted'] = trialData.groupby(['nwb_file_name','epoch'])['contingency'].shift(1)
    trialData['contingency_changed'] = trialData['contingency'] != trialData['contingency_shifted']
    trialData['contingency_count_by_epoch'] = trialData.groupby(['nwb_file_name','epoch'])['contingency_changed'].cumsum()-1
    trialData['trial_number_by_contingency'] = trialData.groupby(['nwb_file_name','epoch', 'contingency']).cumcount()
    trialData['contingency_str'] = trialData['contingency'].astype(str)
    for i in range(1,7):
        trialData[f'p_rew_leaf{i}'] = trialData['contingency_str'].str[(i-1)*2:(i-1)*2+2].astype(int)
    
    return trialData

def find_other_leaf(row):
    if isinstance(row['Sleaves'], list):
        if  str(int(row['leaf_shifted'])) == row['Sleaves'][0]:
            other_leaf = row['Sleaves'][1]
        elif  str(int(row['leaf_shifted'])) == row['Sleaves'][1]:
            other_leaf = row['Sleaves'][0]
        else:
            raise Exception("Warning: looks like leaf_shifted isnt in sleaves 0 or 1?")
    else:
        other_leaf = np.nan
    return other_leaf

def get_log_reg_data_stay_switch_max(data_dict, subject_id):
    trialData = load_trialData(data_dict, subject_id)
    trialData['SstemOption'] = trialData.groupby(by = ['nwb_file_name', 'epoch',])['stem'].shift(1)
    new_cols = trialData.apply(compute_stay_switch_values, axis=1)
    trialData[['Sleaves', 'Gleaves', 'GstemOption', 'SstemOptionBiased', 'GstemOptionBiased']] = pd.DataFrame(new_cols.tolist(), index=trialData.index)

    options_map = {'A': 'QA', 'B': 'QB', 'C': 'QC'}
    options_map_biased = {'1': 'Qstem1', '2': 'Qstem2', '3': 'Qstem3'}
    trialData['qSmean'] = trialData.apply(map_sstemoption, axis=1, args=('SstemOption', options_map))
    trialData['qSmean_biased'] = trialData.apply(map_sstemoption, axis=1, args=('SstemOptionBiased', options_map_biased))

    # For qGmean and qGmean_biased, now also with qGmean_max and qGmean_max_biased
    trialData['qGmean'] = trialData.apply(
        lambda row: np.mean([row['Q' + val] for val in row['GstemOption']]) if isinstance(row['GstemOption'], list) else np.nan,
        axis=1)
    trialData['qGmean_max'] = trialData.apply(
        lambda row: np.max([row['Q' + val] for val in row['GstemOption']]) if isinstance(row['GstemOption'], list) else np.nan,
        axis=1) # this mirrrors softmax a bit more 3 way choice rather than avging
    trialData['qGmean_biased'] = trialData.apply(
        lambda row: np.mean([row['Qstem' + val] for val in row['GstemOptionBiased']]) if isinstance(row['GstemOptionBiased'], list) else np.nan,
        axis=1)
#     trialData['qGmean_max_biased'] = trialData.apply(
#         lambda row: np.max([row['Qstem' + val] for val in row['GstemOptionBiased']]) if isinstance(row['GstemOptionBiased'], list) else np.nan,
#         axis=1)
    
    # now also try to get qSnext as a leaf rather than as a stem avg
    trialData['leaf_shifted'] = trialData.groupby(by = ['nwb_file_name', 'epoch',])['leaf'].shift(1)
    trialData['other_leaf_in_stem'] = trialData.apply(find_other_leaf, axis=1)
    trialData['qSnext'] = trialData.apply(lambda row: row[f'Q{row["other_leaf_in_stem"]}'] if isinstance(row['other_leaf_in_stem'], str) else np.nan, axis=1) #q leaf for the other leaf within the stem that you werent just at
    
    # calculate values for log reg
    trialData['stem_switch'] = (trialData['SstemOption']!=trialData['stem'])
    trialData['SstemOption'][trialData['SstemOption'].isna()] = np.nan
    trialData['qSminusG'] = trialData['qSmean']-trialData['qGmean']
    trialData['qSminusG_next_max'] = trialData['qSnext']-trialData['qGmean_max']
    trialData['qSminusG_biased'] = trialData['qSmean_biased']-trialData['qGmean_biased']
    trialData['not_switch'] = ~trialData['stem_switch']
    # switch-stay instead of stay-switch
    trialData['qGminusS'] = trialData['qGmean']-trialData['qSmean']
    trialData['qGminusS_next_max'] = trialData['qGmean_max'] - trialData['qSnext']
    trialData['qGminusS_biased'] = trialData['qGmean_biased']-trialData['qSmean_biased']
    
    trialData['contingency_shifted'] = trialData.groupby(['nwb_file_name','epoch'])['contingency'].shift(1)
    trialData['contingency_changed'] = trialData['contingency'] != trialData['contingency_shifted']
    trialData['contingency_count_by_epoch'] = trialData.groupby(['nwb_file_name','epoch'])['contingency_changed'].cumsum()-1
    trialData['trial_number_by_contingency'] = trialData.groupby(['nwb_file_name','epoch', 'contingency']).cumcount()
    trialData['contingency_str'] = trialData['contingency'].astype(str)
    for i in range(1,7):
        trialData[f'p_rew_leaf{i}'] = trialData['contingency_str'].str[(i-1)*2:(i-1)*2+2].astype(int)
    
    trialData['p_rew_prior_leaf'] = trialData.apply(lambda row: row[f'p_rew_leaf{row["leaf_shifted"]}'] if isinstance(row['leaf_shifted'], str) else np.nan, axis=1)
    trialData['p_rew_stay_next_leaf'] = trialData.apply(lambda row: row[f'p_rew_leaf{row["other_leaf_in_stem"]}'] if isinstance(row['other_leaf_in_stem'], str) else np.nan, axis=1)
    trialData['p_rew_mean_stay_leaves'] = trialData[['p_rew_prior_leaf','p_rew_stay_next_leaf']].mean(axis=1)
    trialData['p_rew_max_switch_leaf'] = trialData.apply(
        lambda row: np.max([row['p_rew_leaf' + val] for val in row['Gleaves']]) if isinstance(row['Gleaves'], list) else np.nan,
        axis=1)
    trialData['p_rew_mean_switch_leaves'] = trialData.apply(
        lambda row: np.mean([row['p_rew_leaf' + val] for val in row['Gleaves']]) if isinstance(row['Gleaves'], list) else np.nan,
        axis=1)
    trialData['p_rew_max_mean_switch_leaves'] = trialData.apply(
        lambda row: np.max([np.mean([row['p_rew_leaf' + val] for val in row['Gleaves'][0:2]]), np.mean([row['p_rew_leaf' + val] for val in row['Gleaves'][2:]]) ]) if isinstance(row['Gleaves'], list) else np.nan,
        axis=1)
    trialData['p_rew_switch_mean_mins_p_rew_stay_mean'] = trialData['p_rew_mean_switch_leaves'] - trialData['p_rew_mean_stay_leaves']
    trialData['p_rew_switch_mean_mins_p_rew_stay_next'] = trialData['p_rew_mean_switch_leaves'] - trialData['p_rew_stay_next_leaf']
    trialData['p_rew_switch_max_mean_mins_p_rew_stay_mean'] = trialData['p_rew_max_mean_switch_leaves'] - trialData['p_rew_mean_stay_leaves']
    trialData['p_rew_switch_max_mean_mins_p_rew_stay_next'] = trialData['p_rew_max_mean_switch_leaves'] - trialData['p_rew_stay_next_leaf']
    
    trialData['dec_p_rew_stay_next_leaf'] = trialData['p_rew_stay_next_leaf']/100
    trialData['dec_p_rew_mean_stay_leaves'] =  trialData['p_rew_mean_stay_leaves']/100
    trialData['dec_p_rew_max_mean_switch_leaves'] = trialData['p_rew_max_mean_switch_leaves']/100
    trialData['dec_p_rew_switch_max_mean_mins_p_rew_stay_mean'] = trialData['p_rew_switch_max_mean_mins_p_rew_stay_mean']/100
    trialData['dec_p_rew_switch_max_mean_mins_p_rew_stay_next'] = trialData['p_rew_switch_max_mean_mins_p_rew_stay_next']/100
    
    return trialData

def compute_switch_stem_values(row):
    leaves = range(1, 7)
    if row['SstemOption'] == 'A':
        LstemOption = 'B'
        RstemOption = 'C'
        LstemOptionBiased = '2'
        RstemOptionBiased = '3'
    elif row['SstemOption'] == 'B':
        LstemOption = 'C'
        RstemOption = 'A'
        LstemOptionBiased = '3'
        RstemOptionBiased = '1'
    elif row['SstemOption'] == 'C':
        LstemOption = 'A'
        RstemOption = 'B'
        LstemOptionBiased = '1'
        RstemOptionBiased = '2'
    else:
        # Default values if 'SstemOption' is not A, B, or C
        LstemOption, RstemOption, LstemOptionBiased, RstemOptionBiased = np.nan, np.nan, np.nan, np.nan
    return LstemOption, RstemOption, LstemOptionBiased, RstemOptionBiased

def get_data_stem_choice(trialData, subject_id):
    new_cols = trialData.apply(compute_switch_stem_values, axis=1)
    trialData[['LstemOption', 'RstemOption', 'LstemOptionBiased', 'RstemOptionBiased']] = pd.DataFrame(new_cols.tolist(), index=trialData.index)    
    options_map = {'A': 'QA', 'B': 'QB', 'C': 'QC'}
    options_map_biased = {'1': 'Qstem1', '2': 'Qstem2', '3': 'Qstem3'}
    trialData['qLmean_stem'] = trialData.apply(map_sstemoption, axis=1, args=('LstemOption', options_map))
    trialData['qLmean_biased_stem'] = trialData.apply(map_sstemoption, axis=1, args=('LstemOptionBiased', options_map_biased))
    trialData['qRmean_stem'] = trialData.apply(map_sstemoption, axis=1, args=('RstemOption', options_map))
    trialData['qRmean_biased_stem'] = trialData.apply(map_sstemoption, axis=1, args=('RstemOptionBiased', options_map_biased))
    trialData['qLminusR_stem'] = trialData['qLmean_stem'] - trialData['qRmean_stem']
    trialData['qLminusR_biased_stem'] = trialData['qLmean_biased_stem'] - trialData['qRmean_biased_stem']
    trialData['chooseL_stem'] = (trialData['LstemOption'] == trialData['stem'])
    trialData.loc[trialData['not_switch']==True, ['qLminusR_stem', 'chooseL_stem', 'qLminusR_biased_stem']] = np.nan #ignore stay trials
    return trialData

def compute_switch_leaf_values(row):
    leaves = range(1, 7)
    LleafOptionBiased = 1
    RleafOptionBiased = 2
    if row['stem'] == 'A':
        LleafOption = 1
        RleafOption = 2       
    elif row['stem'] == 'B':
        LleafOption = 3
        RleafOption = 4
    elif row['stem'] == 'C':
        LleafOption = 5
        RleafOption = 6
    else:
        # Default values if 'SstemOption' is not A, B, or C
        LleafOption, RleafOption, LleafOptionBiased, RleafOptionBiased = np.nan, np.nan, np.nan, np.nan
    return LleafOption, RleafOption, LleafOptionBiased, RleafOptionBiased

def get_data_leaf_choice(trialData, subject_id):
    new_cols = trialData.apply(compute_switch_leaf_values, axis=1)
    trialData[['LleafOption', 'RleafOption', 'LleafOptionBiased', 'RleafOptionBiased']] = pd.DataFrame(new_cols.tolist(), index=trialData.index)    
    options_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4', 5: 'Q5', 6: 'Q6'}
    options_map_biased = {1: 'Qleaf1', 2: 'Qleaf2'}
    trialData['qL_leaf'] = trialData.apply(map_sstemoption, axis=1, args=('LleafOption', options_map))
    trialData['qL_biased_leaf'] = trialData.apply(map_sstemoption, axis=1, args=('LleafOptionBiased', options_map_biased))
    trialData['qR_leaf'] = trialData.apply(map_sstemoption, axis=1, args=('RleafOption', options_map))
    trialData['qR_biased_leaf'] = trialData.apply(map_sstemoption, axis=1, args=('RleafOptionBiased', options_map_biased))
    trialData['qLminusR_leaf'] = trialData['qL_leaf'] - trialData['qR_leaf']
    trialData['qLminusR_biased_leaf'] = trialData['qL_biased_leaf'] - trialData['qR_biased_leaf']
    trialData['chooseL_leaf'] = (trialData['LleafOption'] == trialData['leaf'])
    trialData.loc[trialData['not_switch']==True, ['qLminusR_leaf', 'chooseL_leaf', 'qLminusR_biased_leaf']] = np.nan #ignore stay trials
    return trialData

def get_log_reg_stay_switch(trialData, subject_id, biased, x_var, y_var, x_min=None, x_max=None):
    trialData_valid = trialData[np.logical_and(trialData[x_var].notna(), trialData[y_var].notna())]

    # Assuming 'x' is your independent variable and 'y' is your binary dependent variable
    X = sm.add_constant(trialData_valid[x_var]) # Adding a constant for the intercept
    y = trialData_valid[y_var]

    model = sm.Logit(y.astype(float), X.astype(float))
    result = model.fit(disp=0)

    # Extract Coefficients, Pseudo R-squared, and p-values
    beta_0 = result.params['const']
    beta_1 = result.params[x_var]
    pseudo_r_squared = result.prsquared
    p_value = result.pvalues[x_var]
    aic = result.aic
    llf = result.llf
    coeffs = [beta_0, beta_1, pseudo_r_squared, p_value, aic, llf]
    

    # Predictions for the Logistic Curve
    if (x_min is not None) & (x_max is not None):
        x_values = np.linspace(x_min, x_max, 300)
    else:
        x_values = np.linspace(trialData_valid[x_var].min(), trialData_valid[x_var].max(), 300)
    y_values = result.predict(sm.add_constant(x_values))
    
    # Equation and Statistics String
    equation_text = f'P = 1 / (1 + e^{{-({beta_0:.2f} + {beta_1:.2f} * x)}})\nPseudo R-squared: {pseudo_r_squared:.3f}\np-value: {p_value:.3e}'
    
    return trialData_valid[x_var], trialData_valid[y_var], x_values, y_values, equation_text, coeffs

def get_glm_stay_switch(trialData, subject_id, biased, x_var1, x_var2, y_var, x_min=None, x_max=None):
    trialData_valid = trialData[np.logical_and(trialData[x_var1].notna(), trialData[y_var].notna()) & trialData[x_var2].notna()]

    # Assuming 'x' is your independent variable and 'y' is your binary dependent variable
    X = sm.add_constant(trialData_valid[[x_var1,x_var2]]) # Adding a constant for the intercept
    y = trialData_valid[y_var]

#     model = sm.Logit(y.astype(float), X.astype(float))
#     result = model.fit(disp=0)
        
    family = sm.families.Binomial()
    model = sm.GLM(y,
                    X,
                   family=family)
    result=model.fit()
    
    
    
    # Extract Coefficients, Pseudo R-squared, and p-values
    beta_0 = result.params['const']
    beta_1 = result.params[x_var1]
    beta_2 = result.params[x_var2]
    pseudo_r_squared = result.pseudo_rsquared(kind='cs')
    p_value = [result.pvalues[x_var1],result.pvalues[x_var2], result.pvalues['const']]
    aic = result.aic
    llf = result.llf
    coeffs = [beta_0, beta_1, beta_2, pseudo_r_squared, p_value, aic, llf]
    

    # Predictions for the Logistic Curve
    if (x_min is not None) & (x_max is not None):
        x_values1 = np.linspace(x_min, x_max, 300)
        x_values2 = np.linspace(x_min, x_max, 300)
        x_values = pd.DataFrame({x_var1:x_values1, x_var2:x_values2})
    else:
        x_values1 = np.linspace(trialData_valid[x_var1].min(), trialData_valid[x_var1].max(), 300)
        x_values2 = np.linspace(trialData_valid[x_var2].min(), trialData_valid[x_var2].max(), 300)
        x_values = pd.DataFrame({x_var1:x_values1, x_var2:x_values2})

    y_values = result.predict(sm.add_constant(x_values))
    
    # Equation and Statistics String
    equation_text = f'equation = f"log(p / (1 - p)) = {np.round(beta_0,2)} + {np.round(beta_1,2)}*X1 + {np.round(beta_2,2)}*X2\nPseudo R-squared: {pseudo_r_squared:.3f}\np-values: {p_value[0]:.3e},{p_value[1]:.3e}'
    
    return trialData_valid[[x_var1,x_var2]], trialData_valid[y_var], x_values, y_values, equation_text, coeffs

def get_glm_stay_switch2(trialData, subject_id, x_var1, x_var2, y_var, x_min=None, x_max=None):
    '''
    
    '''
    trialData_valid = trialData[np.logical_and(trialData[x_var1].notna(), trialData[y_var].notna()) & trialData[x_var2].notna()]

    # Assuming 'x' is your independent variable and 'y' is your binary dependent variable
    X = sm.add_constant(trialData_valid[[x_var1,x_var2]]) # Adding a constant for the intercept
    y = trialData_valid[y_var]

    # Instead of logistic regression w/ a predictor, have two now, so logit link glm binonmial 
    # model = sm.Logit(y.astype(float), X.astype(float))
    # result = model.fit(disp=0)
    
    # Fit model
    family = sm.families.Binomial()
    model = sm.GLM(y,
                   X,
                   family=family)
    result=model.fit()
    
    # Extract Coefficients, Pseudo R-squared, and p-values
    beta_0 = result.params['const']
    beta_1 = result.params[x_var1]
    beta_2 = result.params[x_var2]
    pseudo_r_squared = result.pseudo_rsquared(kind='cs')
    p_value = [result.pvalues[x_var1],result.pvalues[x_var2], result.pvalues['const']]
    p_value0 = result.pvalues['const']
    p_value1 = result.pvalues[x_var1]
    p_value2 = result.pvalues[x_var2]
    aic = result.aic
    llf = result.llf
    coeffs = [beta_0, beta_1, beta_2, pseudo_r_squared, p_value, aic, llf]
    # Also track CIs
    beta_0_lower_ci = result.conf_int().loc['const', 0]
    beta_0_upper_ci = result.conf_int().loc['const', 1]
    beta_1_lower_ci = result.conf_int().loc[x_var1, 0]
    beta_1_upper_ci = result.conf_int().loc[x_var1, 1]
    beta_2_lower_ci = result.conf_int().loc[x_var2, 0]
    beta_2_upper_ci = result.conf_int().loc[x_var2, 1]
    
    coeffs_dict = {"beta_0":beta_0,
                   "beta_1":beta_1,
                   "beta_2":beta_2,
                   "pseudo_r_squared":pseudo_r_squared,
                   "p_value":p_value,
                   "p_value0":p_value0,
                   "p_value1":p_value1,
                   "p_value2":p_value2,
                   "aic":aic,
                   "llf":llf,
                   "beta_0_lower_ci":beta_0_lower_ci,
                   "beta_0_upper_ci":beta_0_upper_ci,
                   "beta_1_lower_ci":beta_1_lower_ci,
                   "beta_1_upper_ci":beta_1_upper_ci,
                   "beta_2_lower_ci":beta_2_lower_ci,
                   "beta_2_upper_ci":beta_2_upper_ci,
                  "n_valid_trials":len(trialData_valid)}

    # Predictions for the Logistic Curve if wanted to plot them
    if (x_min is not None) & (x_max is not None):
        x_values1 = np.linspace(x_min, x_max, 300)
        x_values2 = np.linspace(x_min, x_max, 300)
        x_values = pd.DataFrame({x_var1:x_values1, x_var2:x_values2})
    else:
        x_values1 = np.linspace(trialData_valid[x_var1].min(), trialData_valid[x_var1].max(), 300)
        x_values2 = np.linspace(trialData_valid[x_var2].min(), trialData_valid[x_var2].max(), 300)
        x_values = pd.DataFrame({x_var1:x_values1, x_var2:x_values2})
    y_values = result.predict(sm.add_constant(x_values))
    
    # Equation and Statistics String
    equation_text = f'equation = f"log(p / (1 - p)) = {np.round(beta_0,2)} + {np.round(beta_1,2)}*X1 + {np.round(beta_2,2)}*X2\nPseudo R-squared: {pseudo_r_squared:.3f}\np-values: {p_value[0]:.3e},{p_value[1]:.3e}'
    
    return trialData_valid[[x_var1,x_var2]], trialData_valid[y_var], x_values, y_values, equation_text, coeffs, coeffs_dict

#also from efg
def get_Qleaf_means(data_dict, subject_id):
    trialData = data_dict[subject_id]
    trialData['QA'] = trialData[['Q1','Q2']].mean(axis=1) #np.mean([trialData['Q1'],trialData['Q2']])
    trialData['QB'] = trialData[['Q3','Q4']].mean(axis=1)
    trialData['QC'] = trialData[['Q5','Q6']].mean(axis=1)
    trialData = trialData.reset_index()
    return trialData

#from efg
def get_stay_go_options(row):
    leaves = range(1, 7)
    if row['SstemOption'] == 'A':
        Sleaves = ['1','2']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        GstemOption = ['B','C']
        SstemOptionBiased = '1'
        GstemOptionBiased = ['2','3']
    elif row['SstemOption'] == 'B':
        GstemOption = ['A','C']
        Sleaves = ['3','4']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        SstemOptionBiased = '2'
        GstemOptionBiased = ['1','3']
    elif row['SstemOption'] == 'C':
        GstemOption = ['A','B']
        Sleaves = ['5','6']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        SstemOptionBiased = '3'
        GstemOptionBiased = ['1','2']
    else:
        # Default values if 'SstemOption' is not A, B, or C
        Sleaves, Gleaves, GstemOption, SstemOptionBiased, GstemOptionBiased = np.nan, np.nan, np.nan, np.nan, np.nan
    return Sleaves, Gleaves, GstemOption, SstemOptionBiased, GstemOptionBiased

#efg
# Updated way to "get log reg data" that basically fills in options and options value columns and subtracts stay switch values
def get_option_values(data_dict, subject_id):
    # Find means within patch of unbiased Qleaf values
    trialData = get_Qleaf_means(data_dict, subject_id)
    # Indicate the initial stem (final stem chosen on prior trial)
    trialData['SstemOption'] = trialData.groupby(by = ['nwb_file_name', 'epoch',])['stem'].shift(1)
    # Add stay and go option names, with and without biases
    new_cols = trialData.apply(get_stay_go_options, axis=1)
    trialData[['Sleaves', 'Gleaves', 'GstemOption', 'SstemOptionBiased', 'GstemOptionBiased']] = pd.DataFrame(
        new_cols.tolist(), index=trialData.index)

    # Map stay options to stay option values, both unbiased and biased, to get qSmean and qSmean_biased
    options_map = {'A': 'QA', 'B': 'QB', 'C': 'QC'}
    options_map_biased = {'1': 'Qstem1', '2': 'Qstem2', '3': 'Qstem3'}
    trialData['qSmean'] = trialData.apply(map_sstemoption, axis=1, args=('SstemOption', options_map))
    trialData['qSmean_biased'] = trialData.apply(map_sstemoption, axis=1, args=('SstemOptionBiased', options_map_biased))

    # Map switch optiosn to their values, biased and unbiased, and using max of means as well now
    trialData['qGmean'] = trialData.apply(
        lambda row: np.mean([row['Q' + val] for val in row['GstemOption']]) if isinstance(row['GstemOption'], list) else np.nan,
        axis=1)
    trialData['qGmean_max'] = trialData.apply(
        lambda row: np.max([row['Q' + val] for val in row['GstemOption']]) if isinstance(row['GstemOption'], list) else np.nan,
        axis=1) # this mirrrors softmax a bit more 3 way choice rather than avging
    trialData['qGmean_biased'] = trialData.apply(
        lambda row: np.mean([row['Qstem' + val] for val in row['GstemOptionBiased']]) if isinstance(row['GstemOptionBiased'], list) else np.nan,
        axis=1)
    trialData['qGmean_biased_max'] = trialData.apply(
        lambda row: np.max([row['Qstem' + val] for val in row['GstemOptionBiased']]) if isinstance(row['GstemOptionBiased'], list) else np.nan,
        axis=1)
    
    # Get qSnext as the value of upcoming leaf only
    trialData['leaf_shifted'] = trialData.groupby(by = ['nwb_file_name', 'epoch',])['leaf'].shift(1)
    trialData['other_leaf_in_stem'] = trialData.apply(find_other_leaf, axis=1)
    trialData['qSnext'] = trialData.apply(lambda row: row[f'Q{row["other_leaf_in_stem"]}'] if isinstance(
        row['other_leaf_in_stem'], str) else np.nan, axis=1) #q leaf for the other leaf within the stem that you werent just at
    
    # Find switch and stay trials
    trialData['stem_switch'] = (trialData['SstemOption']!=trialData['stem']) # this is functionally within epoch 
    trialData['SstemOption'][trialData['SstemOption'].isna()] = np.nan # make nans numpy nans
    trialData['not_switch'] = ~trialData['stem_switch']
    
    # Subtract Stay and Go values
    trialData['qSminusG'] = trialData['qSmean']-trialData['qGmean'] # an avg of 2 leaves - avg of 4 leaves
    trialData['qSminusG_next_max'] = trialData['qSnext']-trialData['qGmean_max'] # 1 leaf - avg of 2 leaves
    trialData['qSminusG_biased'] = trialData['qSmean_biased']-trialData['qGmean_biased'] # 1 stem - avg of 2 stems
    # Newer ones
    trialData['qSminusG_mean_max'] = trialData['qSmean']-trialData['qGmean_max'] # avg of 2 leaves - avg of 2 leaves
    trialData['qSminusG_biased_max'] = trialData['qSmean_biased']-trialData['qGmean_biased_max'] # 1 stem - 1 stem
    
    # Switch-stay instead of stay-switch
    trialData['qGminusS'] = trialData['qGmean']-trialData['qSmean']
    trialData['qGminusS_next_max'] = trialData['qGmean_max'] - trialData['qSnext']
    trialData['qGminusS_biased'] = trialData['qGmean_biased']-trialData['qSmean_biased']
    # Newer ones
    trialData['qGminusS_mean_max'] = trialData['qGmean_max']-trialData['qSmean']
    trialData['qGminusS_biased_max'] = trialData['qGmean_biased_max']-trialData['qSmean_biased'] # 1 stem - 1 stem

    # Contingency info
    trialData['contingency_shifted'] = trialData.groupby(['nwb_file_name','epoch'])['contingency'].shift(1)
    trialData['contingency_changed'] = trialData['contingency'] != trialData['contingency_shifted']
    trialData['contingency_count_by_epoch'] = trialData.groupby(['nwb_file_name','epoch'])['contingency_changed'].cumsum()-1
    trialData['trial_number_by_contingency'] = trialData.groupby(['nwb_file_name','epoch', 'contingency']).cumcount()
    trialData['contingency_str'] = trialData['contingency'].astype(str)
    
    # Add individual leaf nominal reward probabilities
    for i in range(1,7):
        trialData[f'p_rew_leaf{i}'] = trialData['contingency_str'].str[(i-1)*2:(i-1)*2+2].astype(int)
    
    # Calculate nominal option values
    trialData['p_rew_prior_leaf'] = trialData.apply(lambda row: row[f'p_rew_leaf{row["leaf_shifted"]}'] if isinstance(
        row['leaf_shifted'], str) else np.nan, axis=1)
    # Stay values nominal
    trialData['p_rew_stay_next_leaf'] = trialData.apply(lambda row: row[f'p_rew_leaf{row["other_leaf_in_stem"]}'] if isinstance(
        row['other_leaf_in_stem'], str) else np.nan, axis=1)
    trialData['p_rew_mean_stay_leaves'] = trialData[['p_rew_prior_leaf','p_rew_stay_next_leaf']].mean(axis=1)
    # Switch values nominal
    trialData['p_rew_max_switch_leaf'] = trialData.apply(
        lambda row: np.max([row['p_rew_leaf' + val] for val in row['Gleaves']]) if isinstance(row['Gleaves'], list) else np.nan,
        axis=1)
    trialData['p_rew_mean_switch_leaves'] = trialData.apply(
        lambda row: np.mean([row['p_rew_leaf' + val] for val in row['Gleaves']]) if isinstance(row['Gleaves'], list) else np.nan,
        axis=1)
    trialData['p_rew_max_mean_switch_leaves'] = trialData.apply(
        lambda row: np.max([np.mean([row['p_rew_leaf' + val] for val in row['Gleaves'][0:2]]), np.mean(
            [row['p_rew_leaf' + val] for val in row['Gleaves'][2:]]) ]) if isinstance(row['Gleaves'], list) else np.nan,
        axis=1)
    
    # Subtract nominal values
    trialData['p_rew_switch_mean_minus_p_rew_stay_mean'] = trialData['p_rew_mean_switch_leaves'] - trialData['p_rew_mean_stay_leaves']
    trialData['p_rew_switch_max_mean_minus_p_rew_stay_mean'] = trialData['p_rew_max_mean_switch_leaves'] - trialData['p_rew_mean_stay_leaves']
    trialData['p_rew_switch_max_mean_minus_p_rew_stay_next'] = trialData['p_rew_max_mean_switch_leaves'] - trialData['p_rew_stay_next_leaf']
    # Scale to 0:1 rather than 0:100
    trialData['dec_p_rew_stay_next_leaf'] = trialData['p_rew_stay_next_leaf']/100
    trialData['dec_p_rew_mean_stay_leaves'] =  trialData['p_rew_mean_stay_leaves']/100
    trialData['dec_p_rew_max_switch_leaf'] = trialData['p_rew_max_switch_leaf']/100
    trialData['dec_p_rew_mean_switch_leaves'] = trialData['p_rew_mean_switch_leaves']/100
    trialData['dec_p_rew_max_mean_switch_leaves'] = trialData['p_rew_max_mean_switch_leaves']/100
    # Subtracted numbers
    trialData['dec_p_rew_switch_mean_minus_p_rew_stay_mean'] = trialData['p_rew_switch_mean_minus_p_rew_stay_mean'] /100
    trialData['dec_p_rew_switch_max_mean_minus_p_rew_stay_mean'] = trialData['p_rew_switch_max_mean_minus_p_rew_stay_mean']/100
    trialData['dec_p_rew_switch_max_mean_minus_p_rew_stay_next'] = trialData['p_rew_switch_max_mean_minus_p_rew_stay_next']/100
    
    return trialData

#from efg
def plot_glm_stay_switch_coefs(all_rat_coeffs,animals,colors,shift_to_positive_vals):
    # ignore all rat data
    # Filter all_rat_coeffs to include only specified rats
    filtered_glm_results = {animal: all_rat_coeffs[animal] for animal in animals}

    # Extract beta coefficients and p-values for the filtered rats
    beta_0 = [results['beta_0'] for results in filtered_glm_results.values()]
    beta_1 = [results['beta_1'] for results in filtered_glm_results.values()]
    beta_2 = [results['beta_2'] for results in filtered_glm_results.values()]
    p_values_0 = [results['p_value0'] for results in filtered_glm_results.values()]
    p_values_1 = [results['p_value1'] for results in filtered_glm_results.values()]
    p_values_2 = [results['p_value2'] for results in filtered_glm_results.values()]

    # Calculate means
    mean_beta_0 = np.mean(beta_0)
    mean_beta_1 = np.mean(beta_1)
    mean_beta_2 = np.mean(beta_2)

    # Create the bar plot
    categories = ['Constant', f'Stay\nValue', f'Switch\nValue']
    mean_values = [mean_beta_0, mean_beta_1, mean_beta_2]

    plt.figure(figsize=(TWO_COLUMN/1.5, TWO_COLUMN/1.2))
    plt.bar(categories, mean_values, color='grey', alpha=0.6, label='All rats')

    # Overlay scatter plot points with jitter
    np.random.seed(41)  # For reproducibility
    jitter_strength = 0.04  # Adjust the strength of jitter here

    for idx, (animal, results) in enumerate(filtered_glm_results.items()):
        jittered_x = [x + np.random.uniform(-jitter_strength, jitter_strength) for x in range(len(categories))]
        beta_values = [results['beta_0'], results['beta_1'], results['beta_2']]
        p_values = [results['p_value0'], results['p_value1'], results['p_value2']]

        for i in range(len(categories)):
            plt.scatter(
                jittered_x[i], 
                beta_values[i], 
                color=colors[idx], 
                edgecolor=colors[idx], 
                facecolor=colors[idx] if p_values[i] <= 0.05 else 'none',
                s=100, 
                label=f'Rat {animal[0].upper()}' if i == 0 else None  # Add label only for the first point to avoid duplicate legend entries
            )

    plt.axhline(0,color='black',lw=1)
    plt.ylabel('Switch Choice Beta Coefficient')
    plt.title(f'Stay: {x_var1}, Switch: {x_var2}, shift to + vals: {shift_to_positive_vals}')
    plt.legend(bbox_to_anchor=(1.01,.9), loc='upper left',frameon=False)
    sns.despine(offset=5)

    # Show plot
    plt.show()

### F1 B

In [ ]:
show_switches=True
show_switch_arrows_top=True
show_switch_arrows_bottom=False

min_n_trials=0
max_n_trials=None
figsize=[TWO_COLUMN+.5,TWO_COLUMN/5.5]

save_fig = False


dates = [1]
epochs = [2]
subject_id = 'j16'
big_df_plot = hmm_results_stable[subject_id] #big_dfs_grouped[subject_id]

dot_size=12
dot_line_thickness=1

aspect_ratio_sq = False

for d in dates:
    plot_date=np.unique(big_df_plot['date'])[d]
    for ep in epochs:
        epochs=[ep]
        try:
            plot_grid_dots_new2(big_df_plot,
                           figsize=figsize,
                           show_switches=show_switches,
                           show_switch_arrows_top=show_switch_arrows_top,
                           show_switch_arrows_bottom=show_switch_arrows_bottom,
                           plot_date=plot_date,
                           epochs=epochs,
                           min_n_trials=min_n_trials,
                           max_n_trials=max_n_trials,
                           save_fig = save_fig,
                           fig_path = fig_path,
                           dot_size = dot_size,
                           dot_line_thickness=dot_line_thickness,
                               aspect_ratio_sq = aspect_ratio_sq)
        except Exception as e:
            print(f"Skipping {plot_date} ep {ep} with exception: {e}")
            pass

### Concat all rat data

In [ ]:
hmm_results_stable['all_rats'] = pd.concat([hmm_results_stable[subject_id] for subject_id in subject_ids])

### F1 D

In [ ]:

save_fig = False

withlegend=True

decimals = 6
colors = iter(cm.tab20b([0,.8, .85, .1, .05]))
fig, axes = plt.subplots(ncols=1, nrows=1, figsize=(TWO_COLUMN/5,(TWO_COLUMN/5)*(1/GOLDEN_RATIO)),  sharex=True)
for subject_id in ['all_rats']: #,'j16', 'chimi', 'senor', 'wilbur','peanut']:
    trialData = get_log_reg_data_stay_switch(hmm_results_stable, subject_id)
    trialData = get_data_stem_choice(trialData,subject_id)
    trialData = get_data_leaf_choice(trialData, subject_id)
    trialData['p_rew_mean_A'] = (trialData['p_rew_leaf1']+trialData['p_rew_leaf2'])/2
    trialData['p_rew_mean_B'] = (trialData['p_rew_leaf3']+trialData['p_rew_leaf4'])/2
    trialData['p_rew_mean_C'] = (trialData['p_rew_leaf5']+trialData['p_rew_leaf6'])/2
    trialData['best_patch_nominal'] = trialData[['p_rew_mean_A','p_rew_mean_B','p_rew_mean_C']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch'] = trialData[['QA', 'QB', 'QC']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch_biased'] = trialData[['Qstem1', 'Qstem2', 'Qstem3']].apply(lambda row: [int(col[-1]) for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['chose_best_patch_nominal'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch_nominal'])]
    trialData['chose_best_patch'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch'])]
    trialData['chose_best_patch_biased'] = [x in y for x,y in zip(trialData['stemchoice'], trialData['best_patch_biased'])]
    
    
#     trialData_grouped = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: pd.Series({
#                 'p_best_patch_nominal': x_df['chose_best_patch_nominal'].sum()/len(x_df),
#                 'p_best_patch': x_df['chose_best_patch'].sum()/len(x_df),
#                 'p_best_patch_biased': x_df['chose_best_patch_biased'].sum()/len(x_df),
#                 'sem_nominal': x_df['chose_best_patch_nominal'].sem(),
#                 'sem_q': x_df['chose_best_patch'].sem(),
#                 'sem_biased': x_df['chose_best_patch_biased'].sem(),})).reset_index()
    
#     nominal = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: x_df['chose_best_patch_nominal'].sum()/len(x_df)).reset_index(name='p_best_patch_nominal')
#     q = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: x_df['chose_best_patch'].sum()/len(x_df)).reset_index(name='p_best_patch')
#     biased = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: x_df['chose_best_patch_biased'].sum()/len(x_df)).reset_index(name='p_best_patch_biased')
    if subject_id!='all_rats':
        alpha=0
        ci=0
        color='grey'
        linestyle='-'
        linewidth=1
    else:
        alpha=1
        ci=95
        color='black'
        linestyle='-'
        linewidth=3
    #axes[0].plot(nominal['trial_number_by_contingency'],nominal['p_best_patch_nominal'] ,alpha=alpha)
    sns.lineplot(data=trialData[trialData.trial_number_by_contingency<=59], x='trial_number_by_contingency', y='chose_best_patch_nominal', ci=ci, err_style='band', ax=axes, alpha=alpha, label='All Rats', linewidth=linewidth, color=color, linestyle=linestyle, err_kws={'edgecolor':None})
#     sns.lineplot(data=trialData, x='trial_number_by_contingency', y='chose_best_patch', ci=ci, err_style='band',  ax=axes[1], alpha=alpha, label=subject_id)
#     sns.lineplot(data=trialData, x='trial_number_by_contingency', y='chose_best_patch_biased', ci=ci, err_style='band', ax=axes[2], alpha=alpha, label=subject_id)
#     axes[1].plot(q['trial_number_by_contingency'],q['p_best_patch'],alpha=alpha)
#     axes[2].plot(biased['trial_number_by_contingency'],biased['p_best_patch_biased'],alpha=alpha)
# for ax in [0]: #,1,2]:
for subject_id in ['j16', 'chimi', 'senor', 'wilbur','peanut']:
    trialData = get_log_reg_data_stay_switch(hmm_results_stable, subject_id)
    trialData = get_data_stem_choice(trialData,subject_id)
    trialData = get_data_leaf_choice(trialData, subject_id)
    trialData['p_rew_mean_A'] = (trialData['p_rew_leaf1']+trialData['p_rew_leaf2'])/2
    trialData['p_rew_mean_B'] = (trialData['p_rew_leaf3']+trialData['p_rew_leaf4'])/2
    trialData['p_rew_mean_C'] = (trialData['p_rew_leaf5']+trialData['p_rew_leaf6'])/2
    trialData['best_patch_nominal'] = trialData[['p_rew_mean_A','p_rew_mean_B','p_rew_mean_C']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch'] = trialData[['QA', 'QB', 'QC']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch_biased'] = trialData[['Qstem1', 'Qstem2', 'Qstem3']].apply(lambda row: [int(col[-1]) for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['chose_best_patch_nominal'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch_nominal'])]
    trialData['chose_best_patch'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch'])]
    trialData['chose_best_patch_biased'] = [x in y for x,y in zip(trialData['stemchoice'], trialData['best_patch_biased'])]
    
    if subject_id!='all_rats':
        alpha=.5
        ci=95
        color=next(colors)
        linestyle='-'
        linewidth=1
    sns.lineplot(data=trialData[trialData.trial_number_by_contingency<=59], x='trial_number_by_contingency', y='chose_best_patch_nominal', ci=0, err_style='band', ax=axes, alpha=alpha, label=f'Rat {subject_id[0].upper() if subject_id != "all_rats" else "All Rats"}', linewidth=linewidth, color=color, linestyle=linestyle, err_kws={'edgecolor':None})

axes.axhline(y=.33, linestyle='--', color='grey', zorder=0,label='Chance')
if withlegend:
    plt.legend(bbox_to_anchor=(1.1, .8), frameon=False,loc='upper left' )

axes.set_xlim(0,59)
axes.set_xticks(ticks = [0,50], ) 
axes.set_ylim(0,1)
axes.set_ylabel('Probability of Choosing\nNominally Best Patch')
axes.set_xlabel('Trials Since Contingency\nBlock Change')

sns.despine(offset = 5)

#axes.legend(bbox_to_anchor=(1.01, .5))
if save_fig:
    fig_name = f'p_chose_best_nominal_patch_post_conting_change_allrats_ci95_ratcolorlines_60trials_legend{withlegend}'
    plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)    
plt.show()

In [ ]:
# Stats

in_patch = 59
total_trials = 100
p=1/3

binomtest(in_patch, total_trials, p=p, alternative='two-sided')

In [ ]:
subject_id = 'all_rats'
trialData = get_log_reg_data_stay_switch(hmm_results_stable, subject_id)
trialData = get_data_stem_choice(trialData,subject_id)
trialData = get_data_leaf_choice(trialData, subject_id)
trialData['p_rew_mean_A'] = (trialData['p_rew_leaf1']+trialData['p_rew_leaf2'])/2
trialData['p_rew_mean_B'] = (trialData['p_rew_leaf3']+trialData['p_rew_leaf4'])/2
trialData['p_rew_mean_C'] = (trialData['p_rew_leaf5']+trialData['p_rew_leaf6'])/2
trialData['best_patch_nominal'] = trialData[['p_rew_mean_A','p_rew_mean_B','p_rew_mean_C']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
trialData['best_patch'] = trialData[['QA', 'QB', 'QC']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
trialData['best_patch_biased'] = trialData[['Qstem1', 'Qstem2', 'Qstem3']].apply(lambda row: [int(col[-1]) for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
trialData['chose_best_patch_nominal'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch_nominal'])]
trialData['chose_best_patch'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch'])]
trialData['chose_best_patch_biased'] = [x in y for x,y in zip(trialData['stemchoice'], trialData['best_patch_biased'])]

trialData['chose_best_patch_nominal']

In [ ]:
k_in_best_patch = trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.sum()
n_trials = len(trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal)
binomtest(k_in_best_patch, n_trials, p=1/3)

In [ ]:
# Stats and plot

save_fig = False

withlegend=True

decimals = 6
colors = iter(cm.tab20b([0,.8, .85, .1, .05]))
fig, axes = plt.subplots(ncols=1, nrows=1, figsize=(TWO_COLUMN/5,(TWO_COLUMN/5)*(1/GOLDEN_RATIO)),  sharex=True)
for subject_id in ['all_rats']: #,'j16', 'chimi', 'senor', 'wilbur','peanut']:
    trialData = get_log_reg_data_stay_switch(hmm_results_stable, subject_id)
    trialData = get_data_stem_choice(trialData,subject_id)
    trialData = get_data_leaf_choice(trialData, subject_id)
    trialData['p_rew_mean_A'] = (trialData['p_rew_leaf1']+trialData['p_rew_leaf2'])/2
    trialData['p_rew_mean_B'] = (trialData['p_rew_leaf3']+trialData['p_rew_leaf4'])/2
    trialData['p_rew_mean_C'] = (trialData['p_rew_leaf5']+trialData['p_rew_leaf6'])/2
    trialData['best_patch_nominal'] = trialData[['p_rew_mean_A','p_rew_mean_B','p_rew_mean_C']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch'] = trialData[['QA', 'QB', 'QC']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch_biased'] = trialData[['Qstem1', 'Qstem2', 'Qstem3']].apply(lambda row: [int(col[-1]) for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['chose_best_patch_nominal'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch_nominal'])]
    trialData['chose_best_patch'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch'])]
    trialData['chose_best_patch_biased'] = [x in y for x,y in zip(trialData['stemchoice'], trialData['best_patch_biased'])]
    
    k_in_best_patch = trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.sum()
    n_trials = trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.count()
    result = binomtest(k_in_best_patch, n_trials, p=1/3)
    result_ci = result.proportion_ci(confidence_level=0.95)
    print(subject_id, result, result_ci)
    
    result = binomtest(k_in_best_patch, n_trials, p=1/3, alternative='greater')
    result_ci = result.proportion_ci(confidence_level=0.95)
    print(subject_id, result, result_ci)
    
    k_in_best_patch = trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.sum()
    n_trials = trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.count()
    result = binomtest(k_in_best_patch, n_trials, p=1/3)
    result_ci = result.proportion_ci(confidence_level=0.95)
    print(subject_id, result, result_ci)
    
    # compare first trial to final trial distribution
    successes = [trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.sum(), trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.sum()]
    nobs = [trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.count(), trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.count()]
    result_compare_proportions = proportions_ztest(successes, nobs)
    print(subject_id, 'two proportion ztest and p val', result_compare_proportions)
    
#     trialData_grouped = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: pd.Series({
#                 'p_best_patch_nominal': x_df['chose_best_patch_nominal'].sum()/len(x_df),
#                 'p_best_patch': x_df['chose_best_patch'].sum()/len(x_df),
#                 'p_best_patch_biased': x_df['chose_best_patch_biased'].sum()/len(x_df),
#                 'sem_nominal': x_df['chose_best_patch_nominal'].sem(),
#                 'sem_q': x_df['chose_best_patch'].sem(),
#                 'sem_biased': x_df['chose_best_patch_biased'].sem(),})).reset_index()
    
#     nominal = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: x_df['chose_best_patch_nominal'].sum()/len(x_df)).reset_index(name='p_best_patch_nominal')
#     q = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: x_df['chose_best_patch'].sum()/len(x_df)).reset_index(name='p_best_patch')
#     biased = trialData.groupby('trial_number_by_contingency').apply(lambda x_df: x_df['chose_best_patch_biased'].sum()/len(x_df)).reset_index(name='p_best_patch_biased')
    if subject_id!='all_rats':
        alpha=0
        ci=0
        color='grey'
        linestyle='-'
        linewidth=1
    else:
        alpha=1
        ci=95
        color='black'
        linestyle='-'
        linewidth=3
    #axes[0].plot(nominal['trial_number_by_contingency'],nominal['p_best_patch_nominal'] ,alpha=alpha)
    sns.lineplot(data=trialData[trialData.trial_number_by_contingency<=59], x='trial_number_by_contingency', y='chose_best_patch_nominal', ci=ci, err_style='band', ax=axes, alpha=alpha, label='All Rats', linewidth=linewidth, color=color, linestyle=linestyle, err_kws={'edgecolor':None})
#     sns.lineplot(data=trialData, x='trial_number_by_contingency', y='chose_best_patch', ci=ci, err_style='band',  ax=axes[1], alpha=alpha, label=subject_id)
#     sns.lineplot(data=trialData, x='trial_number_by_contingency', y='chose_best_patch_biased', ci=ci, err_style='band', ax=axes[2], alpha=alpha, label=subject_id)
#     axes[1].plot(q['trial_number_by_contingency'],q['p_best_patch'],alpha=alpha)
#     axes[2].plot(biased['trial_number_by_contingency'],biased['p_best_patch_biased'],alpha=alpha)
# for ax in [0]: #,1,2]:
for subject_id in ['j16', 'chimi', 'senor', 'wilbur','peanut']:
    trialData = get_log_reg_data_stay_switch(hmm_results_stable, subject_id)
    trialData = get_data_stem_choice(trialData,subject_id)
    trialData = get_data_leaf_choice(trialData, subject_id)
    trialData['p_rew_mean_A'] = (trialData['p_rew_leaf1']+trialData['p_rew_leaf2'])/2
    trialData['p_rew_mean_B'] = (trialData['p_rew_leaf3']+trialData['p_rew_leaf4'])/2
    trialData['p_rew_mean_C'] = (trialData['p_rew_leaf5']+trialData['p_rew_leaf6'])/2
    trialData['best_patch_nominal'] = trialData[['p_rew_mean_A','p_rew_mean_B','p_rew_mean_C']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch'] = trialData[['QA', 'QB', 'QC']].apply(lambda row: [col[-1] for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['best_patch_biased'] = trialData[['Qstem1', 'Qstem2', 'Qstem3']].apply(lambda row: [int(col[-1]) for col in row.index if round(row[col],decimals) == round(row.max(), decimals)], axis=1)
    trialData['chose_best_patch_nominal'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch_nominal'])]
    trialData['chose_best_patch'] = [x in y for x,y in zip(trialData['stem'], trialData['best_patch'])]
    trialData['chose_best_patch_biased'] = [x in y for x,y in zip(trialData['stemchoice'], trialData['best_patch_biased'])]
    
    
    k_in_best_patch = trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.sum()
    n_trials = trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.count()
    result = binomtest(k_in_best_patch, n_trials, p=1/3)
    result_ci = result.proportion_ci(confidence_level=0.95)
    print(subject_id, result, result_ci)
    
    result = binomtest(k_in_best_patch, n_trials, p=1/3, alternative='greater')
    result_ci = result.proportion_ci(confidence_level=0.95)
    print(subject_id, result, result_ci)
    
    k_in_best_patch = trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.sum()
    n_trials = trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.count()
    result = binomtest(k_in_best_patch, n_trials, p=1/3)
    result_ci = result.proportion_ci(confidence_level=0.95)
    print(subject_id, result, result_ci)
    
    # compare first trial to final trial distribution
    successes = [trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.sum(), trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.sum()]
    nobs = [trialData[trialData.trial_number_by_contingency==0].chose_best_patch_nominal.count(), trialData[trialData.trial_number_by_contingency==59].chose_best_patch_nominal.count()]
    result_compare_proportions = proportions_ztest(successes, nobs)
    print(subject_id, 'two proportion ztest and p val', result_compare_proportions)
    
    if subject_id!='all_rats':
        alpha=.5
        ci=95
        color=next(colors)
        linestyle='-'
        linewidth=1
    sns.lineplot(data=trialData[trialData.trial_number_by_contingency<=59], x='trial_number_by_contingency', y='chose_best_patch_nominal', ci=0, err_style='band', ax=axes, alpha=alpha, label=f'Rat {subject_id[0].upper() if subject_id != "all_rats" else "All Rats"}', linewidth=linewidth, color=color, linestyle=linestyle, err_kws={'edgecolor':None})

axes.axhline(y=.33, linestyle='--', color='grey', zorder=0,label='Chance')
if withlegend:
    plt.legend(bbox_to_anchor=(1.1, .8), frameon=False,loc='upper left' )

axes.set_xlim(0,59)
axes.set_xticks(ticks = [0,50], ) 
axes.set_ylim(0,1)
axes.set_ylabel('Probability of Choosing\nNominally Best Patch')
axes.set_xlabel('Trials Since Contingency\nBlock Change')

sns.despine(offset = 5)

#axes.legend(bbox_to_anchor=(1.01, .5))
if save_fig:
    fig_name = f'p_chose_best_nominal_patch_post_conting_change_allrats_ci95_ratcolorlines_60trials_legend{withlegend}'
    plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)    
plt.show()